# Compare email and web notices

Source differences can affect both annotation effort and extraction errors. Compare text lengths, sentence counts, and date forms before choosing model input lengths or feature windows.

The default dataset is synthetic. Its plots check the workflow and do not support research conclusions. Change `DATASET` in the setup cell after collecting and preparing a reviewed corpus.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from smartnotice.demo import run_demo
from smartnotice.io import read_jsonl
from smartnotice.schemas import Notice

%matplotlib inline

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), (
    "Open this notebook from the repository root or notebooks/"
)

DATASET = "demo"  # Set to "research" to read the prepared local corpus.
if DATASET == "demo":
    run_demo(ROOT)
    NOTICE_PATH = ROOT / "data/interim/demo/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/demo/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/examples/events.jsonl"
    PAIR_PATH = ROOT / "data/processed/demo/applicability.jsonl"
else:
    NOTICE_PATH = ROOT / "data/interim/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/adjudicated.jsonl"
    PAIR_PATH = ROOT / "data/processed/applicability.jsonl"

FIGURES = ROOT / "reports/generated" / DATASET / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
notices = [Notice.model_validate(row) for row in read_jsonl(NOTICE_PATH)]
assert notices, "Prepare at least one notice before analysis"
frame = pd.DataFrame([n.model_dump(mode="json") for n in notices])
print(f"Dataset: {DATASET}; documents: {len(notices)}; synthetic: {int(frame.synthetic.sum())}")
if DATASET == "research":
    assert not frame.synthetic.any(), "Exclude synthetic fixtures from research measurements"

In [ ]:
frame["characters"] = frame.text.str.len()
frame["sentences_count"] = frame.sentences.map(len)
frame["numeric_dates"] = frame.text.str.count(r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{4})?\b")
frame["relative_date_mentions"] = frame.text.str.count(r"(?i)tuần này|tuần sau|ngày mai|hôm nay")
frame["table_lines"] = frame.text.map(
    lambda text: sum("; " in line and ": " in line for line in text.splitlines())
)
display(
    frame.groupby("source_type")[
        ["characters", "sentences_count", "numeric_dates", "relative_date_mentions", "table_lines"]
    ].agg(["count", "median", "min", "max"])
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sources = sorted(frame.source_type.unique())
axes[0].boxplot(
    [frame.loc[frame.source_type == source, "characters"] for source in sources],
    tick_labels=sources,
)
axes[1].boxplot(
    [frame.loc[frame.source_type == source, "sentences_count"] for source in sources],
    tick_labels=sources,
)
axes[0].set(title="Notice length by source", ylabel="Unicode characters")
axes[1].set(title="Sentence count by source", ylabel="Sentences")
fig.tight_layout()
fig.savefig(FIGURES / "eda_source_lengths.png", dpi=160)
plt.show()
plt.close("all")

## Inspect date expression coverage

Regex counts describe surface forms. They do not establish extracted dates or normalized-date accuracy. Sentence segmentation uses punctuation and line boundaries, so abbreviations and table formatting need inspection.

In [ ]:
counts = frame.groupby("source_type")[
    ["numeric_dates", "relative_date_mentions", "table_lines"]
].sum()
display(counts)
axis = counts.plot.bar(figsize=(9, 3.8), rot=0, title="Date mentions and table-like lines")
axis.set_ylabel("Count")
axis.figure.tight_layout()
axis.figure.savefig(FIGURES / "eda_date_forms.png", dpi=160)
plt.show()
plt.close("all")

## Inspect a prepared example

The preview below contains prepared text. Complete privacy review before exporting an executed notebook from a real corpus.

In [ ]:
sample = notices[0]
print(sample.doc_id, sample.source_type, sample.reference_date)
for sentence in sample.sentences:
    print(f"{sentence.sentence_id} [{sentence.start}:{sentence.end}] {sentence.text}")

## Write an interpretation

For a reviewed research corpus, connect observed source differences to annotation time, date ambiguity, input length, and candidate relation distances. State sample sizes and link saved figures. Avoid generalizing from the synthetic examples.